In [1]:
from openai import OpenAI
import shelve
import os
import time

In [2]:
OPEN_AI_API_KEY = 'sk-proj-Qjb8HmX7anyLgGATenZaT3BlbkFJ2rxn0QXqk1ldzQmjjSiG'
client = OpenAI(api_key=OPEN_AI_API_KEY)

In [55]:
def upload_file():

    pdf_ch = input("What PDF do you want to select: ")
    pdf_ch = pdf_ch + '.pdf'
    filename = os.path.join('files', pdf_ch)

    # Upload a file with an "assistants" purpose
    file = client.files.create(file=open(filename, "rb"), purpose="assistants")
    return file

In [56]:
file = upload_file()

In [57]:
print(file)

FileObject(id='file-xRvKAgevT4SyHsbi0WbUU4Gd', bytes=1715724, created_at=1718707468, filename='4.pdf', object='file', purpose='assistants', status='processed', status_details=None)


In [3]:
# Create a vector store called "Financial Statements"
try:
    vector_store = client.beta.vector_stores.create(name="Financial Statements")
except Exception as e:
    print(f"Error creating vector store: {e}")
    exit()

# Prompt user for PDF selection
pdf_ch = input("What PDF do you want to select: ")
pdf_ch = pdf_ch + '.pdf'
filename = os.path.join('files', pdf_ch)

# Check if the file exists
if not os.path.isfile(filename):
    print(f"File not found: {filename}")
    exit()

# Ready the files for upload to OpenAI
file_paths = [filename]
try:
    file_streams = [open(pat, "rb") for pat in file_paths]
except Exception as e:
    print(f"Error opening file: {e}")
    exit()

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
try:
    file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id, files=file_streams
    )
except Exception as e:
    print(f"Error during file upload: {e}")
    exit()

# Print the status and the file counts of the batch to see the result of this operation.
print(f"Batch status: {file_batch.status}")
print(f"File counts: {file_batch.file_counts}")

Batch status: failed
File counts: FileCounts(cancelled=0, completed=0, failed=1, in_progress=0, total=1)


In [80]:
def create_assistant(file):
    assistant = client.beta.assistants.create(
        name="PDF Guru",
        instructions="Du er en assistent. Du får en fil som inneholder husdetaljer for en eiendom. Brukeren vil stille deg spørsmål knyttet til dette huset. Du må svare dem deretter. Vær oppmerksom på at hvis du ikke finner noe eksakt svar på det spørsmålet, finn det nærmeste lignende svaret.",
        temperature=1,
        tools=[{"type": "file_search"}],
        model="gpt-4o",
        tool_resources={
        "file_search": {[file.id]
        }
  }
    )
    return assistant

In [81]:
asistant = create_assistant(file)

TypeError: unhashable type: 'list'

In [67]:
print(asistant)

Assistant(id='asst_9IetjGqA99O41gltMuRXEg2R', created_at=1718707544, description=None, instructions='Du er en assistent. Du får en fil som inneholder husdetaljer for en eiendom. Brukeren vil stille deg spørsmål knyttet til dette huset. Du må svare dem deretter. Vær oppmerksom på at hvis du ikke finner noe eksakt svar på det spørsmålet, finn det nærmeste lignende svaret.', metadata={}, model='gpt-3.5-turbo', name='PDF Guru', object='assistant', tools=[CodeInterpreterTool(type='code_interpreter')], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=ToolResourcesCodeInterpreter(file_ids=['file-xRvKAgevT4SyHsbi0WbUU4Gd']), file_search=None), top_p=1.0)


In [61]:
# --------------------------------------------------------------
# Thread management
# --------------------------------------------------------------
def check_if_thread_exists(wa_id):
    with shelve.open("threads_db") as threads_shelf:
        return threads_shelf.get(wa_id, None)


def store_thread(wa_id, thread_id):
    with shelve.open("threads_db", writeback=True) as threads_shelf:
        threads_shelf[wa_id] = thread_id

In [62]:
# --------------------------------------------------------------
# Run assistant
# --------------------------------------------------------------
def run_assistant(thread,asistant):
    # Retrieve the Assistant
    assistant = client.beta.assistants.retrieve(asistant.id)

    # Run the assistant
    run = client.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id,
    )

    # Wait for completion
    while run.status != "completed":
        time.sleep(0.5)
        run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)

    # Retrieve the Messages
    messages = client.beta.threads.messages.list(thread_id=thread.id)
    new_message = messages.data[0].content[0].text.value
    print(f"Generated message: {new_message}")
    return new_message

In [63]:
# --------------------------------------------------------------
# Generate response
# --------------------------------------------------------------
def generate_response(message_body, wa_id, name):
    # Check if there is already a thread_id for the wa_id
    thread_id = check_if_thread_exists(wa_id)

    # If a thread doesn't exist, create one and store it
    if thread_id is None:
        print(f"Creating new thread for {name} with wa_id {wa_id}")
        thread = client.beta.threads.create()
        store_thread(wa_id, thread.id)
        thread_id = thread.id

    # Otherwise, retrieve the existing thread
    else:
        print(f"Retrieving existing thread for {name} with wa_id {wa_id}")
        thread = client.beta.threads.retrieve(thread_id)

    # Add message to thread
    message = client.beta.threads.messages.create(
        thread_id=thread_id,
        role="user",
        content=message_body,
    )

    # Run the assistant and get the new message
    new_message = run_assistant(thread,asistant)
    print(f"To {name}:", new_message)
    return new_message

In [74]:
new_message = generate_response("hva er adressen?","123","Ash")

Retrieving existing thread for Ash with wa_id 123
Generated message: Det ser ut til at det fortsatt oppstår problemer med å ekstrahere tekst fra PDF-filen. Uten mulighet for å se innholdet selv, blir det utfordrende å gi konkret informasjon om adressen.

Jeg foreslår følgende alternativer:
1. **Prøv en annen fil**: Last opp en ny versjon av filen eller en annen fil som kan inneholde den ønskede informasjonen.
2. **Beskriv dokumentet**: Gi mer detaljer om dokumentet, som hvilke sider eller seksjoner adressen kan være i, slik at jeg kan målrette innsatsen bedre.
3. **Sjekk manuelt**: Hvis mulig, kan du sjekke filen manuelt for spesifikk informasjon om adressen.

La meg vite hvordan du ønsker å fortsette!
To Ash: Det ser ut til at det fortsatt oppstår problemer med å ekstrahere tekst fra PDF-filen. Uten mulighet for å se innholdet selv, blir det utfordrende å gi konkret informasjon om adressen.

Jeg foreslår følgende alternativer:
1. **Prøv en annen fil**: Last opp en ny versjon av filen 